<a href="https://colab.research.google.com/github/Holanpasaribu12/UAS/blob/main/UTS_DL_Holan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Restaurant reviews.csv to Restaurant reviews.csv


In [ ]:
# ================================
# 1. INSTALL LIBRARY
# ================================
!pip install pandas numpy scikit-learn tensorflow nltk

# ================================
# 2. IMPORT LIBRARY
# ================================
import pandas as pd
import numpy as np
import re
import nltk

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from google.colab import files

nltk.download('stopwords')
from nltk.corpus import stopwords

# ================================
# 3. UPLOAD DATA
# ================================
uploaded = files.upload()

df = pd.read_csv('Restaurant reviews.csv')

# ================================
# 4. CLEANING DATA
# ================================
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df = df[['Review', 'Rating']]

df = df[df['Review'].notna()]

df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
df['Rating'].fillna(df['Rating'].median(), inplace=True)
df['Rating'] = df['Rating'].astype(int)

print("Jumlah data:", len(df))

# ================================
# 5. PREPROCESSING TEXT
# ================================
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_review'] = df['Review'].apply(clean_text)

# ================================
# 6. LABELING
# ================================
def label_sentiment(rating):
    if rating <= 2:
        return 0
    elif rating == 3:
        return 1
    else:
        return 2

df['label'] = df['Rating'].apply(label_sentiment)

# ================================
# 7. TOKENIZATION
# ================================
max_words = 5000
max_len = 100

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['clean_review'])

X = tokenizer.texts_to_sequences(df['clean_review'])
X = pad_sequences(X, maxlen=max_len)

y = np.array(df['label'])

# ================================
# 8. SPLIT DATA
# ================================
if len(X) > 5:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
else:
    print("Data terlalu sedikit, pakai semua data")
    X_train, X_test, y_train, y_test = X, X, y, y

# ================================
# 9. BUILD MODEL RNN (LSTM)
# ================================
model = Sequential()

model.add(Embedding(input_dim=max_words, output_dim=128))
model.add(LSTM(128))
model.add(Dropout(0.5))

model.add(Dense(64, activation='relu'))
model.add(Dense(3, activation='softmax'))

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

# ================================
# 10. TRAINING
# ================================
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
)

# ================================
# 11. EVALUATION
# ================================
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print("Classification Report:")
print(classification_report(y_test, y_pred_classes))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_classes))

# ================================
# 12. TEST MANUAL
# ================================
def predict_review(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len)
    pred = model.predict(pad)
    label = np.argmax(pred)

    if label == 0:
        return "Negative"
    elif label == 1:
        return "Neutral"
    else:
        return "Positive"

print(predict_review("The food was amazing and the service was excellent"))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Saving Restaurant reviews.csv to Restaurant reviews (4).csv
Jumlah data: 9955


/tmp/ipykernel_12551/1696470273.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Rating'].fillna(df['Rating'].median(), inplace=True)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
249/249 ━━━━━━━━━━━━━━━━━━━━ 49s 183ms/step - accuracy: 0.7337 - loss: 0.6732 - val_accuracy: 0.8187 - val_loss: 0.4872
Epoch 2/5
249/249 ━━━━━━━━━━━━━━━━━━━━ 47s 188ms/step - accuracy: 0.8438 - loss: 0.4133 - val_accuracy: 0.8132 - val_loss: 0.5053
Epoch 3/5
249/249 ━━━━━━━━━━━━━━━━━━━━ 45s 182ms/step - accuracy: 0.8820 - loss: 0.3191 - val_accuracy: 0.7921 - val_loss: 0.5330
Epoch 4/5
249/249 ━━━━━━━━━━━━━━━━━━━━ 46s 185ms/step - accuracy: 0.9125 - loss: 0.2450 - val_accuracy: 0.7931 - val_loss: 0.6845
Epoch 5/5
249/249 ━━━━━━━━━━━━━━━━━━━━ 46s 184ms/step - accuracy: 0.9337 - loss: 0.1915 - val_accuracy: 0.7921 - val_loss: 0.6935
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step
Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.81      0.80       509
           1       0.35      0.33      0.34       253
           2       0.88      0.88      0.88      1229

    accuracy                           0.79      1991
   macro avg   